# Regularization — Implementations

Ridge and lasso again, with the two things worth carrying away: the α↔λ translation between penalty conventions (a factor of n for ridge, a factor of 2 for lasso — both silent when wrong), and ISTA as what you do when half the objective has no gradient.

## 03_ridge

L2: shrink everything a little.

### torch

The same closed form on tensors. **What torch adds:** a warning, mostly — `weight_decay=λ` in any torch optimiser penalises (λ/2)‖θ‖² under this notebook's convention, so the closed form is the honest lane.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Centre X and y first; the intercept is recovered afterwards, unpenalised.
# 2. The notebook's objective is (1/n)||Xθ−y||² + λ||θ||², so the solve adds n·λ·I.
# 3. torch.linalg.solve on (XcᵀXc + nλI) — a linear solve, not an inverse.
# 4. optim.SGD(weight_decay=λ) would implement +（λ/2)||θ||² — a factor of 2 off.


class RidgeRegressionScratch:
    """Ridge in closed form on tensors, same centering trick and same convention:
    L(θ) = (1/n)‖Xθ − y‖² + λ‖θ‖², intercept unpenalised."""

    def __init__(self, lam=1.0):
        self.lam = lam

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        yt = torch.as_tensor(np.asarray(y, dtype=float))
        self.x_mean_ = Xt.mean(dim=0)
        self.y_mean_ = float(yt.mean())
        Xc = Xt - self.x_mean_
        yc = yt - self.y_mean_
        n, p = Xc.shape
        A = Xc.T @ Xc + n * self.lam * torch.eye(p, dtype=Xc.dtype)
        coef = torch.linalg.solve(A, Xc.T @ yc)
        self.coef_ = coef.numpy()
        self.intercept_ = float(self.y_mean_ - self.x_mean_ @ coef)
        return self

    def predict(self, X):
        return self.intercept_ + np.asarray(X) @ self.coef_


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(11)
X_eq = _rng_eq.normal(size=(80, 3))
y_eq = X_eq @ np.array([2.0, -1.0, 0.5]) + 4.0 + _rng_eq.normal(0, 0.1, size=80)

_fit_eq = RidgeRegressionScratch(lam=0.5).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("coef:", np.round(coef, 5))


In [ ]:
# Shrinkage is monotone in λ: more penalty, smaller slope norm.
_norms = [np.linalg.norm(RidgeRegressionScratch(lam=l).fit(X_eq, y_eq).coef_)
          for l in (0.0, 0.5, 5.0, 50.0)]
assert all(a > b for a, b in zip(_norms, _norms[1:])), "‖coef‖ must shrink as λ grows"
assert np.linalg.norm(coef) < _norms[0], "λ=0.5 shrinks relative to OLS"


### library

sklearn's `Ridge(alpha=n·λ)` — the translation is the entire content of this lane, and it is exact.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge

# hints:
# 1. sklearn minimises ||Xθ−y||² + α||θ||² — no 1/n on the data term.
# 2. So α = n·λ reproduces the notebook's (1/n)-convention exactly.
# 3. Forgetting that factor is the classic silent bug when λ came from a paper.


class RidgeRegressionScratch:
    """sklearn's Ridge, with the α = n·λ translation made explicit rather than
    left for a bad day. Same attributes as the scratch lanes."""

    def __init__(self, lam=1.0):
        self.lam = lam

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self._model = Ridge(alpha=X.shape[0] * self.lam).fit(X, np.asarray(y, dtype=float))
        self.coef_ = self._model.coef_
        self.intercept_ = float(self._model.intercept_)
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X))


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(11)
X_eq = _rng_eq.normal(size=(80, 3))
y_eq = X_eq @ np.array([2.0, -1.0, 0.5]) + 4.0 + _rng_eq.normal(0, 0.1, size=80)

_fit_eq = RidgeRegressionScratch(lam=0.5).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("coef:", np.round(coef, 5))


In [ ]:
# The translation is exact, not approximate: α = n·λ gives the same estimator,
# so this lane must agree with the closed-form lanes to solver precision.
_ols_like = RidgeRegressionScratch(lam=0.0).fit(X_eq, y_eq)
_resid = y_eq - _ols_like.predict(X_eq)
assert np.max(np.abs(X_eq.T @ _resid)) < 1e-6, "λ=0 must reduce to OLS"


## 03_lasso

L1: push some of it to exactly zero.

### torch

Proximal gradient (ISTA): autograd differentiates the smooth half, `soft_thresh` handles the ‖θ‖₁ half that has no derivative at 0. **What torch adds:** the clean split between the two halves.

In [ ]:
import numpy as np
import torch

# hints:
# 1. ISTA: a gradient step on the smooth part, then soft-threshold the result.
# 2. The step size is 1/L with L = 2·λmax(XcᵀXc)/n — power-iterate or eigh for it.
# 3. prox of t·λ‖·‖₁ is soft_thresh(·, t·λ): the threshold scales with the step.
# 4. Coordinate descent and ISTA reach the same minimiser; the path differs.


def soft_thresh(z, t):
    """The proximal operator of the ℓ₁ norm — where the zeros come from."""
    return torch.sign(z) * torch.clamp(torch.abs(z) - t, min=0.0)


class LassoRegressionScratch:
    """Lasso by proximal gradient (ISTA) on tensors. Same objective as the
    notebook — L(θ) = (1/n)‖Xθ − y‖² + λ‖θ‖₁ — reached by a different route:
    autograd handles the smooth half, soft_thresh handles the half autograd
    cannot differentiate. That split is the whole idea of proximal methods."""

    def __init__(self, lam=0.1, n_sweeps=500, tol=1e-7):
        self.lam = lam
        self.n_sweeps = n_sweeps
        self.tol = tol

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        yt = torch.as_tensor(np.asarray(y, dtype=float))
        self.x_mean_ = Xt.mean(dim=0)
        self.y_mean_ = float(yt.mean())
        Xc = Xt - self.x_mean_
        yc = yt - self.y_mean_
        n, p = Xc.shape

        # Lipschitz constant of ∇ smooth part: L = 2·λmax(XcᵀXc)/n.
        L = 2.0 * float(torch.linalg.eigvalsh(Xc.T @ Xc)[-1]) / n
        step = 1.0 / L

        theta = torch.zeros(p, dtype=Xc.dtype, requires_grad=True)
        self.n_sweeps_used_ = 0
        for sweep in range(1, self.n_sweeps + 1):
            smooth = torch.mean((Xc @ theta - yc) ** 2)
            smooth.backward()
            with torch.no_grad():
                new_theta = soft_thresh(theta - step * theta.grad, step * self.lam)
                max_change = float(torch.max(torch.abs(new_theta - theta)))
                theta.copy_(new_theta)
                theta.grad = None
            self.n_sweeps_used_ = sweep
            if max_change < self.tol:
                break

        self.coef_ = theta.detach().numpy()
        self.intercept_ = float(self.y_mean_ - self.x_mean_.numpy() @ self.coef_)
        return self

    def predict(self, X):
        return self.intercept_ + np.asarray(X) @ self.coef_

    def active_set(self, tol=1e-8):
        return np.where(np.abs(self.coef_) > tol)[0]


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(23)
X_eq = _rng_eq.normal(size=(120, 8))
_theta_eq = np.array([3.0, -2.0, 1.5, 0.0, 0.0, 0.0, 0.0, 0.0])
y_eq = X_eq @ _theta_eq + 1.0 + _rng_eq.normal(0, 0.3, size=120)

_fit_eq = LassoRegressionScratch(lam=0.2, n_sweeps=20000, tol=1e-13).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("active:", _fit_eq.active_set(), "sweeps:", _fit_eq.n_sweeps_used_)


In [ ]:
assert set(_fit_eq.active_set(1e-6)) == {0, 1, 2}, "the three real features, and only them"
assert np.max(np.abs(coef[3:])) < 1e-6, "the five noise features are exactly zeroed"


### library

sklearn's `Lasso(alpha=λ/2)` — its data term carries a 1/(2n), so the penalty translates by a factor of two.

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso

# hints:
# 1. sklearn minimises (1/2n)||Xθ−y||² + α||θ||₁ — note the extra 1/2.
# 2. Multiplying the notebook's objective by 1/2 shows α = λ/2 is the translation.
# 3. Get it wrong and every coefficient is shrunk as if λ were doubled.


class LassoRegressionScratch:
    """sklearn's Lasso, α = λ/2 because of its (1/2n) data-term convention."""

    def __init__(self, lam=0.1, n_sweeps=500, tol=1e-7):
        self.lam = lam
        self.n_sweeps = n_sweeps
        self.tol = tol

    def fit(self, X, y):
        self._model = Lasso(alpha=self.lam / 2, max_iter=self.n_sweeps,
                            tol=self.tol).fit(np.asarray(X), np.asarray(y))
        self.coef_ = self._model.coef_
        self.intercept_ = float(self._model.intercept_)
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X))

    def active_set(self, tol=1e-8):
        return np.where(np.abs(self.coef_) > tol)[0]


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(23)
X_eq = _rng_eq.normal(size=(120, 8))
_theta_eq = np.array([3.0, -2.0, 1.5, 0.0, 0.0, 0.0, 0.0, 0.0])
y_eq = X_eq @ _theta_eq + 1.0 + _rng_eq.normal(0, 0.3, size=120)

_fit_eq = LassoRegressionScratch(lam=0.2, n_sweeps=20000, tol=1e-13).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_


In [ ]:
assert set(_fit_eq.active_set(1e-6)) == {0, 1, 2}, "same support as every other lane"
